# MSigDB decoupleR ORA Analysis: pathway coverage across study sizes

**Environment:** `clamp-analyses`

For each CLAMP model (CLAMPfull and CLAMPbase) across study coverage levels, this notebook:

1. Loads the Z matrix (gene loadings per LV) for a given study coverage level/seed.
2. Filters genes to the model gene universe overlapping MSigDB (v2026.1).
3. Runs `decoupleR::run_ora()` on the full loading matrix at once (all LVs as columns), selecting the top 1% positive-loading genes per LV (`n_up`), no bottom/negative tail (`n_bottom = 0`), with `n_background` set to the filtered model gene universe size.
4. Stores raw `terms_padj`: the minimum BH-adjusted p-value per MSigDB term across all LVs (BH-adjustment done within each LV, then minimum taken across LVs — same convention as the clusterProfiler-based `00_bp_ora_analysis.ipynb` sibling notebook).
5. Saves per-seed RDS caches (`rs{pct}_seed{seed}_msigdb_decoupler_ora.rds`) and per-pct-level summary RDS/CSV. FDR thresholds and coverage computation are done in `01_msigdb_decoupler_ora_plot.ipynb`.

In [ ]:
library(here)
library(dplyr)
library(decoupleR)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
output_dir <- here("output/03_model_biology/00_archs4/06_coverage_study/decoupler_ora")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"),  recursive = TRUE, showWarnings = FALSE)

## Model specs: discovered dynamically from disk

In [ ]:
study_subdirs <- list.dirs(models_dir, recursive = FALSE, full.names = TRUE)

model_specs <- do.call(c, lapply(study_subdirs, function(study_dir) {
  seed_dirs <- list.dirs(study_dir, recursive = FALSE, full.names = TRUE)
  specs <- lapply(seed_dirs, function(seed_dir) {
    bn <- basename(seed_dir)
    m  <- regmatches(bn, regexec("^study_coverage_rs([0-9]+)_seed_([0-9]+)$", bn))[[1]]
    if (length(m) < 3) return(NULL)
    z_path <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
    if (!file.exists(z_path)) return(NULL)
    list(
      coverage = as.integer(m[2]),
      dir      = basename(study_dir),
      seed     = as.integer(m[3]),
      seed_dir = seed_dir
    )
  })
  Filter(Negate(is.null), specs)
}))

model_specs <- model_specs[order(
  sapply(model_specs, `[[`, "coverage"),
  sapply(model_specs, `[[`, "seed")
)]

message("Found ", length(model_specs), " models:")
for (s in model_specs) {
  message(sprintf("  rs%d%% seed%d: %s", s$coverage, s$seed, s$dir))
}

## Load MSigDB gene sets as a decoupleR network

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
net <- msig_gmt %>% dplyr::rename(source = term, target = gene)
message(sprintf("MSigDB gene sets loaded: %d", length(unique(net$source))))

## Helper: run decoupleR ORA for one model

Returns a list with raw `terms_padj` (minimum BH-adjusted p-value per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  mat            <- as.matrix(Z[universe_genes %in% net$target, , drop = FALSE])
  n_background   <- nrow(mat)
  n_top          <- ceiling(0.01 * n_background)
  n_lvs          <- ncol(mat)

  term_overlap   <- tapply(net$target %in% rownames(mat), net$source, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  ora_res <- decoupleR::run_ora(
    mat          = mat,
    network      = net,
    n_up         = n_top,
    n_bottom     = 0,
    n_background = n_background,
    minsize      = 10
  )

  # BH-adjust within each LV (condition), then take min adjusted p per pathway across LVs
  ora_res <- ora_res %>%
    dplyr::group_by(condition) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
    dplyr::ungroup()

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(ora_res$p_adj, ora_res$source, min)
  )
}

## Run ORA: CLAMPfull

In [ ]:
coverage_values <- sort(unique(sapply(model_specs, `[[`, "coverage")))
results_clampfull_by_pct <- list()

for (cov in coverage_values) {
  cov_specs    <- Filter(function(s) s$coverage == cov, model_specs)
  pct_rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb_decoupler_ora.rds", cov))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPfull %d%%", cov))
    results_clampfull_by_pct[[as.character(cov)]] <- readRDS(pct_rds_path)
    next
  }

  cov_rows <- lapply(cov_specs, function(spec) {
    z_path     <- file.path(spec$seed_dir, "CLAMPfull_hall", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPfull",
                            sprintf("rs%d_seed%d_msigdb_decoupler_ora.rds", spec$coverage, spec$seed))

    sub_info_path <- file.path(spec$seed_dir, "subsample_info.rds")
    n_studies <- NA_integer_
    if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPfull rs%d seed%d", spec$coverage, spec$seed))
      res <- readRDS(cache_path)
      if (is.null(res$n_studies)) res$n_studies <- n_studies
    } else {
      message(sprintf("Running decoupleR ORA: CLAMPfull rs%d seed%d", spec$coverage, spec$seed))
      res <- run_ora_for_model(z_path)
      if (!is.null(res)) {
        res$n_studies <- n_studies
        saveRDS(res, cache_path)
      }
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPfull",
      coverage_pct   = spec$coverage,
      seed           = spec$seed,
      n_studies      = res$n_studies,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), cov_rows))
  rownames(pct_df) <- NULL
  results_clampfull_by_pct[[as.character(cov)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPfull %d%% -> %s", cov, pct_rds_path))
}

results_clampfull_df <- do.call(rbind, results_clampfull_by_pct)
rownames(results_clampfull_df) <- NULL
print(results_clampfull_df)

In [ ]:
for (cov in names(results_clampfull_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%s_msigdb_decoupler_ora.csv", cov))
  write.csv(results_clampfull_by_pct[[cov]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}

## Run ORA: CLAMPbase

In [ ]:
results_clampbase_by_pct <- list()

for (cov in coverage_values) {
  cov_specs    <- Filter(function(s) s$coverage == cov, model_specs)
  pct_rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb_decoupler_ora.rds", cov))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPbase %d%%", cov))
    results_clampbase_by_pct[[as.character(cov)]] <- readRDS(pct_rds_path)
    next
  }

  cov_rows <- lapply(cov_specs, function(spec) {
    z_path     <- file.path(spec$seed_dir, "CLAMPbase", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPbase",
                            sprintf("rs%d_seed%d_msigdb_decoupler_ora.rds", spec$coverage, spec$seed))

    sub_info_path <- file.path(spec$seed_dir, "subsample_info.rds")
    n_studies <- NA_integer_
    if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPbase rs%d seed%d", spec$coverage, spec$seed))
      res <- readRDS(cache_path)
      if (is.null(res$n_studies)) res$n_studies <- n_studies
    } else {
      message(sprintf("Running decoupleR ORA: CLAMPbase rs%d seed%d", spec$coverage, spec$seed))
      res <- run_ora_for_model(z_path)
      if (!is.null(res)) {
        res$n_studies <- n_studies
        saveRDS(res, cache_path)
      }
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPbase",
      coverage_pct   = spec$coverage,
      seed           = spec$seed,
      n_studies      = res$n_studies,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), cov_rows))
  rownames(pct_df) <- NULL
  results_clampbase_by_pct[[as.character(cov)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPbase %d%% -> %s", cov, pct_rds_path))
}

results_clampbase_df <- do.call(rbind, results_clampbase_by_pct)
rownames(results_clampbase_df) <- NULL
print(results_clampbase_df)

In [ ]:
for (cov in names(results_clampbase_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%s_msigdb_decoupler_ora.csv", cov))
  write.csv(results_clampbase_by_pct[[cov]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}